# Base-rate merged results

Explore `data/base_rate/base_rate_merged_results.csv` from a benchmark run.

Each row has **`score`** (`true`/`false`): whether the parsed answer matches **`scepticism_score_target`**. Unparseable rows have `score=false` and `parseable=false`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "base_rate").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MERGED_DIR = ROOT / "data" / "base_rate"
MERGED_CSV = None
for name in (
    # "base_rate_merged_results.csv",
    "base_rate_merged_results (5).csv",
):
    candidate = MERGED_DIR / name
    if candidate.is_file():
        MERGED_CSV = candidate
        break
if MERGED_CSV is None:
    raise FileNotFoundError(
        f"Missing merged results under {MERGED_DIR}. Run the base-rate benchmark first "
        "(benchmark/base-rate-benchmark.ipynb)."
    )

df = pd.read_csv(MERGED_CSV)

if "score" in df.columns:
    df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
elif "score_outcome" in df.columns:
    df["score_value"] = (df["score_outcome"] == "normative").astype(int)
else:
    raise KeyError("Merged CSV must include 'score' or legacy 'score_outcome'.")

if "parseable" in df.columns:
    df["parseable_bool"] = df["parseable"].astype(str).str.lower().eq("true")

df["score_true"] = df["score_value"].astype(bool)

print("Loaded:", MERGED_CSV)
print("Rows:", len(df))
print("Models:", sorted(df["model"].unique()))
print("Vignettes:", df["vignette_name"].nunique())
df.head()

Loaded: c:\src2\sceptical-llms\data\base_rate\base_rate_merged_results (5).csv
Rows: 34
Models: ['google/gemini-2.5-flash']
Vignettes: 10


,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,parsed_answer_type,parsed_percent,parsed_choice,parsed_confidence,scoring_type,parseable,score,score_value,parseable_bool,score_true
0,actor_waiter_overlap__overlap__implausible__mc...,actor waiter overlap,overlap,small,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is...,False,implausible,...,mc_choice,NaN,A,5,mc_full,True,False,0,True,False
1,actor_waiter_overlap__overlap__mc_full_probs,actor waiter overlap,overlap,small,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is...,False,underdetermined,...,mc_choice,NaN,B,5,mc_full,True,False,0,True,False
2,actor_waiter_overlap__overlap__mc_numeric_probs,actor waiter overlap,overlap,small,mc_numeric,True,mc_numeric_probs,You are a statistical consultant. Your task is...,False,underdetermined,...,mc_choice,NaN,B,5,mc_numeric,True,False,0,True,False
3,actor_waiter_overlap__overlap__open_probs,actor waiter overlap,overlap,small,open,True,open_probs,You are a statistical consultant. Your task is...,False,underdetermined,...,probability,0.26,NaN,5,open,True,True,1,True,True
4,ca_trump_voter__implausible__mc_full_probs,CA Trump voter,well_posed,0,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is...,True,implausible,...,mc_choice,NaN,C,5,mc_full,True,False,0,True,False


In [2]:
df.columns

Index(['example_id', 'vignette_name', 'problem_type', 'intersection_size',
       'response_type', 'has_statistics', 'variant', 'prompt', 'well_posed',
       'normative', 'p_c_and_d_given_a', 'normative_choice',
       'normative_percent', 'normative_open', 'confidence_required',
       'numeric_score_percent', 'numeric_score_choice', 'scepticism_required',
       'scepticism_score_target', 'option_a_label', 'option_b_label',
       'option_c_label', 'option_d_label', 'option_e_label', 'option_a_lure',
       'option_b_lure', 'option_c_lure', 'option_d_lure', 'option_e_lure',
       'option_f_label', 'option_g_label', 'option_h_label', 'option_f_lure',
       'option_g_lure', 'option_h_lure', 'model', 'llm_response', 'reasoning',
       'answer_line', 'confidence_line', 'parsed_answer_type',
       'parsed_percent', 'parsed_choice', 'parsed_confidence', 'scoring_type',
       'parseable', 'score', 'score_value', 'parseable_bool', 'score_true'],
      dtype='str')

In [3]:
df['problem_type'].value_counts()

problem_type
well_posed    20
overlap       14
Name: count, dtype: int64

In [4]:
  df['variant'].value_counts()

variant
mc_full_probs       20
mc_numeric_probs     7
open_probs           7
Name: count, dtype: int64

In [5]:
 df['intersection_size'].value_counts()

intersection_size
0         20
small      8
large      4
medium     2
Name: count, dtype: int64

## `mc_numeric_probs` detail

For each vignette: MC options A–E, the model's letter (`parsed_choice`), partition shortcut letter (`numeric_score_choice`), scepticism fields, and score.

In [6]:
MC_NUMERIC_CHOICE_COLS = [f"option_{letter}_label" for letter in "abcde"]


def format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter, col in zip("ABCDE", MC_NUMERIC_CHOICE_COLS):
        value = row.get(col)
        if pd.notna(value) and str(value).strip():
            parts.append(f"{letter}: {value}")
    return " | ".join(parts)


mc_numeric_probs = df[df["variant"] == "mc_numeric_probs"].copy()
mc_numeric_probs["choices_offered"] = mc_numeric_probs.apply(format_mc_choices, axis=1)

mc_numeric_probs_view = mc_numeric_probs[
    [
        "vignette_name",
        "choices_offered",
        "parsed_choice",
        "numeric_score_choice",
        "scepticism_required",
        "scepticism_score_target",
        "score",
        "score_value",
        "normative_choice",
        "answer_line",
    ]
].sort_values("vignette_name")

pd.set_option("display.max_colwidth", 140)
mc_numeric_probs_view

,vignette_name,choices_offered,parsed_choice,numeric_score_choice,scepticism_required,scepticism_score_target,score,score_value,normative_choice,answer_line
6,CA Trump voter,A: About 10% | B: About 1% | C: About 4% | D: About 6% | E: About 13%,D,A,False,NaN,False,0,A,D
2,actor waiter overlap,A: About 0% | B: About 19%,B,A,False,NaN,False,0,A,B
12,covid vaccine (blue/red),A: About 20% | B: About 8% | C: About 14% | D: About 0% | E: About 27%,C,A,False,NaN,False,0,A,C
18,discharged weapon (last year),A: About 91% | B: About 0% | C: About 70% | D: About 89% | E: About 44%,A,A,False,NaN,True,1,A,A
24,healthcare employment,A: About 88% | B: About 40% | C: About 26% | D: About 87% | E: About 11%,D,A,False,NaN,False,0,A,D
28,military overseas (federal pool),A: About 82% | B: About 73% | C: About 36% | D: About 63% | E: About 40%,A,A,False,NaN,True,1,A,A
32,professional drivers speeding,A: About 3% | B: About 0% | C: About 2% | D: About 10%,A,A,False,NaN,True,1,A,A


## `open_probs` detail

Benchmark scoring fields plus re-parsed response values. Each `llm_response` is re-parsed (strip trailing confidence, extract all % / 0–1 decimals). **`score_true`** is whether any candidate is within ±0.5 pp of **`scepticism_score_target`**.

In [7]:
import sys
from pathlib import Path

if "ROOT" not in globals():
    ROOT = Path.cwd()
    if not (ROOT / "data" / "base_rate").is_dir():
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks.base_rate import (
    load_benchmark,
    matches_scepticism_target,
    parse_open_response,
    strip_trailing_confidence,
)

benchmark_items = load_benchmark()


def rescore_open_row(row: pd.Series) -> pd.Series:
    item = benchmark_items[row["example_id"]]
    parsed = parse_open_response(str(row["llm_response"]))
    body, confidence = strip_trailing_confidence(str(row["llm_response"]))
    score_true = matches_scepticism_target(item, parsed)
    return pd.Series(
        {
            "response_body": body,
            "parsed_confidence": confidence,
            "parsed_numbers": list(parsed.percent_candidates),
            "parsed_percent_rescored": parsed.percent,
            "parsed_answer_type_rescored": parsed.answer_type,
            "parseable_rescored": parsed.answer_type != "unparseable",
            "score_true": score_true,
        }
    )


open_probs = df[df["variant"] == "open_probs"].copy()
open_probs = open_probs.drop(columns=["score_true"], errors="ignore")
open_probs = pd.concat([open_probs, open_probs.apply(rescore_open_row, axis=1)], axis=1)

# Write rescored values back into the main frame for open_probs rows.
idx = open_probs.index
df.loc[idx, "parsed_percent"] = pd.to_numeric(open_probs["parsed_percent_rescored"], errors="coerce")
df.loc[idx, "parsed_answer_type"] = open_probs["parsed_answer_type_rescored"].astype("string")
df.loc[idx, "parseable"] = open_probs["parseable_rescored"].astype(bool)
df.loc[idx, "parseable_bool"] = open_probs["parseable_rescored"].astype(bool)
df.loc[idx, "score"] = open_probs["score_true"].astype(bool)
df.loc[idx, "score_value"] = open_probs["score_true"].astype(int)
df.loc[idx, "score_true"] = open_probs["score_true"].astype(bool)

OPEN_PROBS_SCORING_COLUMNS = [
    "example_id",
    "vignette_name",
    "normative",
    "scepticism_required",
    "normative_percent",
    "normative_open",
    "numeric_score_percent",
    "scepticism_score_target",
    "parsed_numbers",
    "parsed_percent_rescored",
    "parsed_answer_type_rescored",
    "parsed_confidence",
    "parseable_rescored",
    "score_true",
]

open_probs_view = (
    open_probs[OPEN_PROBS_SCORING_COLUMNS]
    .sort_values(["vignette_name", "normative"])
    .reset_index(drop=True)
)

print(
    "Rescored open_probs score_true:",
    int(open_probs["score_true"].sum()),
    "/",
    len(open_probs),
)
pd.set_option("display.max_colwidth", 120)
open_probs_view

Rescored open_probs score_true: 1 / 7


,example_id,vignette_name,normative,scepticism_required,normative_percent,normative_open,numeric_score_percent,scepticism_score_target,parsed_numbers,parsed_percent_rescored,parsed_answer_type_rescored,parsed_confidence,parsed_confidence,parseable_rescored,score_true
0,ca_trump_voter__open_probs,CA Trump voter,well_posed,False,9.91800,9.9%,9.91200,9.918,[20.5],20.50,probability,5,5,True,False
1,actor_waiter_overlap__overlap__open_probs,actor waiter overlap,underdetermined,False,0.03737,0.04%,0.03966,0.03737,"[0.26, 26.0]",0.26,probability,5,5,True,True
2,covid_vaccine_blue_red__open_probs,covid vaccine (blue/red),well_posed,False,19.62000,20%,19.62000,19.62,[22.5],22.50,probability,4,4,True,False
3,discharged_weapon_last_year__open_probs,discharged weapon (last year),well_posed,False,91.21000,91%,91.21000,91.21,[98.24],98.24,probability,5,5,True,False
4,healthcare_employment__open_probs,healthcare employment,well_posed,False,88.01000,88%,88.01000,88.01,[98.7],98.70,probability,5,5,True,False
5,military_overseas_federal_pool__open_probs,military overseas (federal pool),well_posed,False,81.54000,82%,81.52000,81.54,[90.13],90.13,probability,5,5,True,False
6,professional_drivers_speeding__overlap__open_probs,professional drivers speeding,underdetermined,False,2.64800,2.6%,2.65800,2.648,[5.5],5.50,probability,5,5,True,False


In [8]:
open_probs_responses = (
    open_probs[
        ["vignette_name", "normative_open", "parsed_numbers", "llm_response"]
    ]
    .sort_values("vignette_name")
    .reset_index(drop=True)
)

with pd.option_context("display.max_colwidth", None, "display.width", None):
    display(open_probs_responses)

,vignette_name,normative_open,parsed_numbers,llm_response
0,CA Trump voter,9.9%,[20.5],20.5%\n5
1,actor waiter overlap,0.04%,"[0.26, 26.0]",0.26%\n5
2,covid vaccine (blue/red),20%,[22.5],22.5%\n4
3,discharged weapon (last year),91%,[98.24],98.24%\n5
4,healthcare employment,88%,[98.7],98.7%\n5
5,military overseas (federal pool),82%,[90.13],90.13%\n5
6,professional drivers speeding,2.6%,[5.5],5.5%\n5


## `mc_full_probs` detail

Benchmark scoring fields plus re-parsed MC choice (bottom-up scan for A–H), **only rows with `scepticism_required=true`**. **`score`** is whether **`parsed_choice_rescored`** matches **`scepticism_score_target`** (single letter, or any of `F|G|H`).

In [18]:
from benchmarks.base_rate import (
    load_benchmark,
    matches_scepticism_target,
    parse_response,
)

if "benchmark_items" not in globals():
    benchmark_items = load_benchmark()


def rescore_mc_full_row(row: pd.Series) -> pd.Series:
    item = benchmark_items[row["example_id"]]
    parsed = parse_response(str(row["llm_response"]), scoring_type="mc_full")
    score_true = matches_scepticism_target(item, parsed)
    return pd.Series(
        {
            "answer_line_rescored": parsed.answer_line,
            "parsed_confidence": parsed.confidence,
            "parsed_choice_rescored": parsed.choice,
            "parsed_answer_type_rescored": parsed.answer_type,
            "parseable_rescored": parsed.choice is not None,
            "score": score_true,
        }
    )


mc_full_probs = df[df["variant"] == "mc_full_probs"].copy()
mc_full_probs = mc_full_probs.drop(columns=["score"], errors="ignore")
mc_full_probs = pd.concat(
    [mc_full_probs, mc_full_probs.apply(rescore_mc_full_row, axis=1)],
    axis=1,
)

MC_FULL_PROBS_SCORING_COLUMNS = [
    "example_id",
    "vignette_name",
    "normative",
    "scepticism_required",
    "normative_choice",
    "numeric_score_choice",
    "scepticism_score_target",
    "parsed_choice_rescored",
    "parsed_answer_type_rescored",
    "answer_line_rescored",
    "parsed_confidence",
    "parseable_rescored",
    "score",
]

mc_full_probs_view = (
    mc_full_probs.loc[
        mc_full_probs["scepticism_required"].astype(str).str.lower().eq("true"),
        MC_FULL_PROBS_SCORING_COLUMNS,
    ]
    .sort_values(["vignette_name", "normative"])
    .reset_index(drop=True)
)

print(
    "mc_full_probs score (scepticism_required):",
    int(mc_full_probs_view["score"].sum()),
    "/",
    len(mc_full_probs_view),
)

pd.set_option("display.max_colwidth", 120)
mc_full_probs_view.head(n=5)

mc_full_probs score: 1 / 20


,example_id,vignette_name,normative,scepticism_required,normative_choice,numeric_score_choice,scepticism_score_target,parsed_choice_rescored,parsed_answer_type_rescored,answer_line_rescored,parsed_confidence,parsed_confidence,parseable_rescored,score
0,ca_trump_voter__implausible__mc_full_probs,CA Trump voter,implausible,True,A,A,H,C,mc_choice,C,5,5,True,False
1,ca_trump_voter__mc_full_probs,CA Trump voter,well_posed,False,A,A,A,C,mc_choice,C,5,5,True,False
2,actor_waiter_overlap__overlap__implausible__mc_full_probs,actor waiter overlap,implausible,True,A,A,H,A,mc_choice,A,5,5,True,False
3,actor_waiter_overlap__overlap__mc_full_probs,actor waiter overlap,underdetermined,False,A,A,A,B,mc_choice,B,5,5,True,False
4,college_stem_work__overlap__implausible__mc_full_probs,college STEM work,implausible,True,A,B,H,B,mc_choice,B,4,4,True,False


In [10]:
df['reasoning'].value_counts(), df['confidence_required'].value_counts(), df['normative_choice'].value_counts()

(Series([], Name: count, dtype: int64),
 confidence_required
 True    34
 Name: count, dtype: int64,
 normative_choice
 A    27
 Name: count, dtype: int64)

## Scores by `response_type`

In [11]:
RESPONSE_TYPE_ORDER = ["open", "mc_numeric", "mc_full"]


def score_summary_table(group_col: str, *, order: list[str] | None = None) -> pd.DataFrame:
    """Counts, parseability mix, and mean score for each group value."""
    work = df.copy()
    if "parseable_bool" not in work.columns:
        work["parseable_bool"] = True
    work["score_miss"] = work["parseable_bool"] & (work["score_value"] == 0)
    work["unparseable_row"] = ~work["parseable_bool"]

    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "score_true": grouped["score_value"].sum(),
            "score_false": grouped["score_miss"].sum(),
            "unparseable": grouped["unparseable_row"].sum(),
            "score_rate": grouped["score_value"].mean(),
        }
    )
    summary["score_pct"] = (summary["score_rate"] * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])

    return summary


by_response_type = score_summary_table("response_type", order=RESPONSE_TYPE_ORDER)
by_response_type

,n,score_true,score_false,unparseable,score_rate,score_pct
response_type,,,,,,
open,7,1,6,0,0.142857,14.3
mc_numeric,7,3,4,0,0.428571,42.9
mc_full,20,1,19,0,0.050000,5.0


## Scores by `variant`

In [12]:
VARIANT_ORDER = [
    "open_probs",
    "mc_numeric_probs",
    "mc_full_probs",
]

by_variant = score_summary_table("variant", order=VARIANT_ORDER)
by_variant

,n,score_true,score_false,unparseable,score_rate,score_pct
variant,,,,,,
open_probs,7,1,6,0,0.142857,14.3
mc_numeric_probs,7,3,4,0,0.428571,42.9
mc_full_probs,20,1,19,0,0.050000,5.0


## Scores by `vignette_name`

In [13]:
by_vignette = score_summary_table(
    "vignette_name",
    order=sorted(df["vignette_name"].unique()),
)
by_vignette

,n,score_true,score_false,unparseable,score_rate,score_pct
vignette_name,,,,,,
CA Trump voter,4,0,4,0,0.00,0.0
actor waiter overlap,4,1,3,0,0.25,25.0
college STEM work,2,0,2,0,0.00,0.0
covid vaccine (blue/red),4,0,4,0,0.00,0.0
diabetes insulin obese,2,0,2,0,0.00,0.0
discharged weapon (last year),4,1,3,0,0.25,25.0
english teacher humanities,2,0,2,0,0.00,0.0
healthcare employment,4,0,4,0,0.00,0.0
military overseas (federal pool),4,1,3,0,0.25,25.0


## Optional: split by model when multiple LLMs are present

In [14]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "response_type"], observed=True)["score_value"]
        .mean()
        .unstack("response_type")
        .reindex(columns=RESPONSE_TYPE_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["score_value"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")

Single model in file — see tables above.


In [15]:
df['model'].value_counts()

model
google/gemini-2.5-flash    34
Name: count, dtype: int64

In [16]:
Trump = df.query("vignette_name == 'CA Trump voter'")
Trump

,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,parsed_answer_type,parsed_percent,parsed_choice,parsed_confidence,scoring_type,parseable,score,score_value,parseable_bool,score_true
4,ca_trump_voter__implausible__mc_full_probs,CA Trump voter,well_posed,0,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,True,implausible,...,mc_choice,NaN,C,5,mc_full,True,False,0,True,False
5,ca_trump_voter__mc_full_probs,CA Trump voter,well_posed,0,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,True,well_posed,...,mc_choice,NaN,C,5,mc_full,True,False,0,True,False
6,ca_trump_voter__mc_numeric_probs,CA Trump voter,well_posed,0,mc_numeric,True,mc_numeric_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,True,well_posed,...,mc_choice,NaN,D,5,mc_numeric,True,False,0,True,False
7,ca_trump_voter__open_probs,CA Trump voter,well_posed,0,open,True,open_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\n...,True,well_posed,...,probability,20.5,NaN,5,open,True,False,0,True,False
